Настройка окружения

In [ ]:
import os

from dotenv import load_dotenv

# Импорт основных компонентов
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

загружаем ключ из .env

In [ ]:
load_dotenv()

API_KEY = (os.getenv("GIGA_KEY") or "").strip()
giga_scope = (os.getenv("GIGACHAT_SCOPE") or "GIGACHAT_API_PERS").strip()

if not API_KEY or API_KEY == "ваш_ключ":
    raise ValueError("Укажите реальный GIGA_KEY в .env")

Убедись, что все работает

In [ ]:
# Настройка модели GigaChat с расширенными параметрами
llm = GigaChat(
    credentials=API_KEY,
    scope=giga_scope,
    model="GigaChat-Pro",  # можно указать конкретную версию, по умолчанию Lite
    verify_ssl_certs=False,
    temperature=0.1,  # настройка креативности
    max_tokens=1000,  # максимальная длина ответа
)

In [ ]:
# Проверка подключения
response = llm.invoke("Привет! Как дела?")
print(response.content)

## Задача: извлечение количества проживающих из заявок

Компания по аренде жилья получает тысячи заявок в виде неструктурированного текста.  
Нужно автоматически извлекать ключевую информацию — **количество проживающих**.

**Анализ сложностей:**
- вариативность формулировок количества людей;
- неявные указания («семья с двумя детьми» = 4 человека);
- отсутствие прямого указания количества в некоторых текстах;
- необходимость возвращения именно **целочисленного** результата.

### Базовое промптирование

In [ ]:
# Простой промпт для извлечения количества людей
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:""",
)

# Создание цепочки
chain = basic_prompt | llm | StrOutputParser()

In [ ]:
# Индивидуальная работа: 15 заявок из rental_04.csv (вариант 04)
import csv

test_texts = {}
with open("rental_04.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    for i, row in enumerate(reader, start=1):
        if i > 15:
            break
        test_texts[i] = row["text"].strip()

for idx, text in test_texts.items():
    result = chain.invoke({"text": text})
    print(f"#{idx}")
    print(f"Текст: {text}")
    print(f"Результат: {result}")
    print("---")

### Загрузка в DataFrame

`df` хранит все заявки из файла.
- столбец `text` содержит текст заявки;
- столбец `amount` хранит правильный ответ.

In [ ]:
import pandas as pd

# Загрузка данных из csv в датафрейм df
df = pd.read_csv("rental_04.csv", sep=";")

# Просмотр первых 5 строк
df.head()

### Обработка заявок в цикле и сохранение результатов

Проходим по столбцу `text`, отправляем каждую заявку в модель, 
сохраняем ответы в новый столбец `result` и записываем всю таблицу в CSV.

In [ ]:
results = []

for _, row in df.iterrows():
    text = row["text"]  # текст заявки
    try:
        result = chain.invoke({"text": text})
        results.append(result)
    except Exception as e:
        results.append(f"ERROR: {e}")

df["result"] = results

df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")

### Оценка точности модели

Сравниваем правильные ответы `amount` и предсказания модели `result`,
считаем количество ошибок и точность в процентах.

In [ ]:
print(df.dtypes)

result_num = pd.to_numeric(df["result"], errors="coerce")
amount_num = pd.to_numeric(df["amount"], errors="coerce")

correct_mask = result_num == amount_num

total = len(df)
correct = int(correct_mask.sum())
errors = int((~correct_mask).sum())
accuracy = correct / total if total else 0.0

print(f"Ошибок: {errors}")
print(f"Точность: {accuracy:.1%}")